
## Speculative Feature Engineering

### Objective

The objective is to exploit the possibility that the dataset was generated using an underlying synthetic process with structured mathematical relationships.

The feature transformations are motivated by three competition resources:

1. [First-place solution write-up](https://www.kaggle.com/competitions/playground-series-s6e1/writeups/1st-place-ive-ran-out-of-catchy-phrases-v)
2. [RealMLP solution and validation results](https://www.kaggle.com/code/mahoganybuttstrings/pg-s6e1-realmlp-cv-8-58748-lb-8-58006)
3. [Competition discussion](https://www.kaggle.com/competitions/playground-series-s6e1/discussion/665915)

These resources motivate additional formulas that may represent latent nonlinear relationships more directly than the original features.

The approach is intentionally speculative: the transformations are hypotheses about the data-generation process, not proof that the underlying generating function has been recovered.


In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
from src.backend import is_cuda_available

device = "cuda" if is_cuda_available() else "cpu"

device

'cuda'

In [3]:
numerical_features = ["age", "study_hours", "class_attendance", "sleep_hours"]
categorical_features = [
    "gender",
    "course",
    "internet_access",
    "sleep_quality",
    "study_method",
    "facility_rating",
    "exam_difficulty",
]

In [4]:
import pandas as pd
import numpy as np


def create_features(df: pd.DataFrame):
    X = df.copy()

    for p in [12, 14, 20]:
        X[f"_study_hours_sin_{p}"] = np.sin(2 * np.pi * X["study_hours"] / p).astype(
            "float32"
        )
        X[f"_class_attendance_sin_{p}"] = np.sin(
            2 * np.pi * X["class_attendance"] / p
        ).astype("float32")
        X[f"_study_hours_cos_{p}"] = np.cos(2 * np.pi * X["study_hours"] / p).astype(
            "float32"
        )

    for c in numerical_features:
        X[f"{c}_sq"] = X[c] ** 2

    X["feature_formula"] = (
        5.9051154511950499 * X["study_hours"]
        + 0.34540967058057986 * X["class_attendance"]
        + 1.423461171860262 * X["sleep_hours"]
        + 4.7819
    )

    X["_feature_formula"] = (
        6 * X["study_hours"]
        + 0.35 * X["class_attendance"]
        + 1.5 * X["sleep_hours"]
        + 5 * (X["sleep_quality"] == "good")
        + -5 * (X["sleep_quality"] == "poor")
        + 10 * (X["study_method"] == "coaching")
        + 5 * (X["study_method"] == "mixed")
        + 2 * (X["study_method"] == "group study")
        + 1 * (X["study_method"] == "online videos")
        + 4 * (X["facility_rating"] == "high")
        + -4 * (X["facility_rating"] == "low")
    )

    for c in ["age"]:
        for k in range(-1, 1):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    for c in ["study_hours"]:
        for k in range(0, 3):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    for c in ["class_attendance"]:
        for k in range(-1, 2):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    for c in ["sleep_hours"]:
        for k in range(0, 2):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    return X


### Engineered Features

The additional transformations include periodic representations of `study_hours` using sine and cosine functions with candidate periods of 12, 14, and 20.

For a period $T$, the corresponding transformations are

$$
f_{\sin}(x;T)
=
\sin\left(\frac{2\pi x}{T}\right),
$$

$$
f_{\cos}(x;T)
=
\cos\left(\frac{2\pi x}{T}\right).
$$

These features allow a model to represent cyclic patterns that cannot be expressed as directly through the raw value alone.

Additional formulas inspired by the referenced solutions are included to expose candidate interactions and nonlinear transformations that may approximate the synthetic process more closely.

The periods and formulas are exploratory assumptions. Their usefulness is assessed empirically through the final competition scores rather than assumed in advance.


In [5]:
train = pd.read_csv("../train.csv")
test = pd.read_csv("../test.csv")

train.drop("id", axis=1, inplace=True)

In [6]:
train_fe = create_features(train)
test_fe = create_features(test)

In [7]:
y_train = train_fe["exam_score"]
X_train: pd.DataFrame = train_fe.drop("exam_score", axis=1)
X_test: pd.DataFrame = test_fe

In [8]:
X_train.head()

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,...,age_d-1,age_d0,study_hours_d0,study_hours_d1,study_hours_d2,class_attendance_d-1,class_attendance_d0,class_attendance_d1,sleep_hours_d0,sleep_hours_d1
0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,...,2,1,7,9,1,9,8,8,4,9
1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,...,1,8,4,9,5,9,4,8,4,7
2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,...,2,0,4,6,8,9,2,6,5,8
3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,...,1,9,2,0,0,4,9,5,8,3
4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,...,2,3,7,6,5,8,6,9,9,6


In [9]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
from catboost import CatBoostRegressor


def catboost_fit(
    model: CatBoostRegressor,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    model.set_params(early_stopping_rounds=50)

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        verbose=False,
        cat_features=categorical_features,
    )

    return model


def catboost_predict(model: CatBoostRegressor, X_valid: pd.DataFrame):
    return model.predict(X_valid)

In [ ]:
from src.predict import ensemble_predict

train, oof_pred_df, test_pred_df = ensemble_predict(
    {
        "CatBoost": (
            CatBoostRegressor(
                iterations=2000,
                depth=6,
                loss_function="RMSE",
                eval_metric="RMSE",
                random_seed=42,
                thread_count=-1,
                verbose=False,
                allow_writing_files=False,
                task_type="GPU" if device == "cuda" else "CPU",
            ),
            catboost_fit,
            catboost_predict,
            False,
            True,
        ),
    },
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    cv=kf,
    numerical_features=[],
    checkpoint_dir="../speculative",
    recompute=False,
    ignore=[],
)

[checkpoint] CatBoost: loaded



### Model Choice and Training Strategy

CatBoost is selected because it can handle categorical variables natively, avoiding the need for explicit one-hot or ordinal encoding.

The model is fitted on the full training dataset using the engineered feature set and an increased number of boosting iterations (`iterations=2000`). It is then used to generate predictions for the competition test set.

This experiment evaluates the combined effect of the speculative transformations and the increased boosting budget. Since both are changed together, the final score alone cannot isolate the individual contribution of each change.


In [15]:
submission = X_test[["id"]]
submission["exam_score"] = test_pred_df["CatBoost"]
submission.to_csv("../speculative.csv", index=False)


### Baseline Submission

The CatBoost model trained on the original feature set achieves:

| Evaluation | RMSE |
|:--|--:|
| Public leaderboard | **8.72428** |
| Private leaderboard | 8.74838 |

These results establish the leaderboard baseline for evaluating the additional transformations.

Because the model is fitted on the full training dataset, these scores should be compared with other leaderboard submissions rather than treated as directly equivalent to the local OOF metric.



### Final Submission Results

Two final approaches were evaluated on the Kaggle leaderboard: a hill-climbing ensemble combining eight model families and a CatBoost model trained with speculative feature engineering.

| Approach | Public RMSE | Private RMSE |
|:--|--:|--:|
| Hill-climbing ensemble | 8.68173 | 8.71180 |
| CatBoost with original features | 8.72428 | 8.74838 |
| **CatBoost with engineered features** | **8.62019** | **8.65025** |

The hill-climbing ensemble improves upon the original-feature CatBoost baseline on both leaderboard splits. Its OOF RMSE of 8.72311 also represents a modest improvement over the best standalone XGBoost model at 8.73149.

However, the largest improvement comes from the speculative feature engineering experiment. Relative to the original-feature CatBoost submission, the engineered-feature model reduces RMSE by:

$$
\Delta_{\text{public}}
=
8.72428-8.62019
=
0.10409,
$$

$$
\Delta_{\text{private}}
=
8.74838-8.65025
=
0.09813.
$$

The improvement is consistent across the public and private leaderboards, suggesting that the engineered features provide useful predictive information beyond the original representation.

Compared with the hill-climbing ensemble, the final CatBoost model also achieves lower RMSE on both splits:

$$
\Delta_{\text{public}}
=
8.68173-8.62019
=
0.06154,
$$

$$
\Delta_{\text{private}}
=
8.71180-8.65025
=
0.06155.
$$

This comparison highlights the relative impact of the two approaches. Ensembling provides a modest improvement by combining models with different inductive biases, whereas the speculative feature engineering appears to expose more useful structure in the dataset.

**Final decision:** use CatBoost with the engineered features and 2,000 boosting iterations for the final submission. It achieves the best result among the evaluated submissions, with a public RMSE of **8.62019** and a private RMSE of **8.65025**.

Although the results suggest that the engineered transformations are well suited to this dataset, they do not establish that the underlying synthetic data-generation process has been exactly reconstructed.
